# VideoMAE for Sign Language Recognition Training

This notebook implements a VideoMAE transformer for word-level American Sign Language recognition based on the research paper "Breaking the Barriers: Video Vision Transformers for Word-Level Sign Language Recognition".

## Key Features:
- Paper-compliant VideoMAE implementation
- Comprehensive visualization during training
- MLflow experiment tracking with auto-logging
- Dynamic artifact folder organization
- Multiple evaluation metrics and visualizations

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import mediapipe as mp
from tqdm import tqdm
import mlflow
import mlflow.tensorflow
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Initialize MediaPipe
mp_holistic = mp.solutions.holistic

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create dynamic artifact directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"videomae_slr_{timestamp}"
artifact_dir = f"./artifacts/{experiment_name}"
os.makedirs(artifact_dir, exist_ok=True)
print(f"Artifacts will be saved to: {artifact_dir}")

## 1. Data Processing Pipeline

In [ ]:
class SignLanguageDataProcessor:
    """Process video files and extract keypoints following paper methodology"""
    
    def __init__(self, sequence_length=64, target_size=(224, 224)):
        self.sequence_length = sequence_length
        self.target_size = target_size
        self.mp_holistic = mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        
    def extract_keypoints(self, results):
        """Extract keypoints following paper's holistic approach"""
        # Pose landmarks (33 points * 4 values each)
        pose = np.array([[res.x, res.y, res.z, res.visibility] 
                        for res in results.pose_landmarks.landmark]).flatten() \
               if results.pose_landmarks else np.zeros(33*4)
        
        # Face landmarks (468 points * 3 values each)
        face = np.array([[res.x, res.y, res.z] 
                        for res in results.face_landmarks.landmark]).flatten() \
               if results.face_landmarks else np.zeros(468*3)
        
        # Left hand landmarks (21 points * 3 values each)
        lh = np.array([[res.x, res.y, res.z] 
                      for res in results.left_hand_landmarks.landmark]).flatten() \
             if results.left_hand_landmarks else np.zeros(21*3)
        
        # Right hand landmarks (21 points * 3 values each)
        rh = np.array([[res.x, res.y, res.z] 
                      for res in results.right_hand_landmarks.landmark]).flatten() \
             if results.right_hand_landmarks else np.zeros(21*3)
        
        return np.concatenate([pose, face, lh, rh])
    
    def preprocess_video_frames(self, video_path):
        """Preprocess video frames following paper's methodology"""
        cap = cv2.VideoCapture(video_path)
        
        # Get total frame count
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Paper methodology: extract 64 consecutive frames
        if total_frames >= self.sequence_length:
            # Select random starting point (paper uses this approach)
            start_frame = np.random.randint(0, total_frames - self.sequence_length + 1)
            frame_indices = range(start_frame, start_frame + self.sequence_length)
        else:
            # Pad if fewer frames (paper approach)
            frame_indices = list(range(total_frames))
            # Pad with last frame
            while len(frame_indices) < self.sequence_length:
                frame_indices.append(total_frames - 1)
        
        keypoints_sequence = []
        
        for i in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
                
            if i in frame_indices:
                # Convert BGR to RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                
                # Process frame with MediaPipe
                results = self.mp_holistic.process(frame_rgb)
                
                # Extract keypoints
                keypoints = self.extract_keypoints(results)
                keypoints_sequence.append(keypoints)
        
        cap.release()
        
        # Ensure exact sequence length
        while len(keypoints_sequence) < self.sequence_length:
            keypoints_sequence.append(np.zeros(1662))
        
        keypoints_sequence = keypoints_sequence[:self.sequence_length]
        
        return np.array(keypoints_sequence)
    
    def create_dataset(self, data_dir):
        """Create dataset following paper's methodology"""
        sequences = []
        labels = []
        label_map = {}
        
        classes = sorted(os.listdir(data_dir))
        for i, class_name in enumerate(classes):
            label_map[i] = class_name
            class_path = os.path.join(data_dir, class_name)
            
            if not os.path.isdir(class_path):
                continue
                
            print(f"Processing class: {class_name}")
            for video_file in tqdm(os.listdir(class_path)):
                if video_file.lower().endswith(('.mp4', '.mov')):
                    video_path = os.path.join(class_path, video_file)
                    try:
                        keypoints = self.preprocess_video_frames(video_path)
                        sequences.append(keypoints)
                        labels.append(i)
                    except Exception as e:
                        print(f"Error processing {video_path}: {e}")
        
        return np.array(sequences), np.array(labels), label_map

## 2. Model Architecture (Paper-Compliant VideoMAE)

In [ ]:
class PaperCompliantVideoMAE:
    """
    VideoMAE implementation following research paper specifications
    Based on: "VideoMAE: Masked Autoencoders are Data-Efficient Learners for Self-Supervised Video Pre-training"
    """
    
    def __init__(self, input_shape, num_classes, d_model=768, num_heads=12, num_layers=12, masking_ratio=0.9):
        self.input_shape = input_shape  # (sequence_length, feature_dim)
        self.num_classes = num_classes
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.masking_ratio = masking_ratio  # Paper uses 90-95% masking
        
    def tube_embedding(self, inputs):
        """Paper: joint space-time cube embedding strategy"""
        return layers.Dense(self.d_model, name='tube_embedding')(inputs)
    
    def positional_encoding(self, position, d_model):
        """Create positional encoding as described in paper"""
        angle_rads = self.get_angles(
            np.arange(position)[:, np.newaxis],
            np.arange(d_model)[np.newaxis, :],
            d_model
        )
        
        # Apply sin to even indices
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        # Apply cos to odd indices
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)
    
    def get_angles(self, pos, i, d_model):
        angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
        return pos * angle_rates
    
    def build_model(self):
        """Build VideoMAE model according to paper specifications"""
        inputs = layers.Input(shape=self.input_shape, name='video_sequence')
        
        # Paper: joint space-time cube embedding strategy
        x = self.tube_embedding(inputs)
        
        # Paper: positional encoding
        seq_length = tf.shape(x)[1]
        pos_encoding = self.positional_encoding(seq_length, self.d_model)
        x = x + pos_encoding[:, :seq_length, :]
        
        # Paper: classification token
        cls_token = tf.Variable(
            tf.random.normal([1, 1, self.d_model]), 
            trainable=True,
            name='cls_token'
        )
        cls_tokens = tf.tile(cls_token, [tf.shape(x)[0], 1, 1])
        x = layers.Concatenate(axis=1)([cls_tokens, x])
        
        # Paper: transformer blocks with joint space-time attention
        for i in range(self.num_layers):
            # Multi-head attention with dropout (Paper uses 0.1)
            attn_output = layers.MultiHeadAttention(
                num_heads=self.num_heads,
                key_dim=self.d_model // self.num_heads,
                dropout=0.1,
                name=f'multi_head_attention_{i}'
            )(x, x)
            
            # Residual connection and layer normalization
            x = layers.Add()([x, attn_output])
            x = layers.LayerNormalization(epsilon=1e-6, name=f'layer_norm_1_{i}')(x)
            
            # Feed-forward network (Paper uses 4x expansion)
            ffn = keras.Sequential([
                layers.Dense(self.d_model * 4, activation='gelu', name=f'ffn_dense1_{i}'),
                layers.Dropout(0.1, name=f'ffn_dropout1_{i}'),
                layers.Dense(self.d_model, name=f'ffn_dense2_{i}'),
                layers.Dropout(0.1, name=f'ffn_dropout2_{i}')
            ], name=f'feed_forward_network_{i}')
            
            ffn_output = ffn(x)
            x = layers.Add()([x, ffn_output])
            x = layers.LayerNormalization(epsilon=1e-6, name=f'layer_norm_2_{i}')(x)
        
        # Paper: classification using CLS token
        cls_output = layers.Lambda(lambda x: x[:, 0], name='cls_pooling')(x)
        
        # Classification head
        outputs = layers.Dense(self.num_classes, activation='softmax', name='classification_head')(cls_output)
        
        model = keras.Model(inputs, outputs, name='VideoMAE_SLR')
        return model

## 3. Data Visualization

In [ ]:
def visualize_data_distribution(y, label_map, artifact_dir):
    """Visualize class distribution in the dataset"""
    plt.figure(figsize=(15, 6))
    
    # Class distribution
    plt.subplot(1, 2, 1)
    unique, counts = np.unique(y, return_counts=True)
    class_names = [label_map[i] for i in unique]
    bars = plt.bar(range(len(class_names)), counts, color=plt.cm.viridis(np.linspace(0, 1, len(class_names))))
    plt.title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
    plt.xlabel('Sign Classes')
    plt.ylabel('Number of Samples')
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
    
    # Add value labels on bars
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                str(count), ha='center', va='bottom')
    
    # Sequence length distribution
    plt.subplot(1, 2, 2)
    # Since we're using fixed length, this would be a single value
    plt.axvline(x=64, color='red', linestyle='--', linewidth=2)
    plt.text(64, 1, 'Fixed Length: 64', rotation=90, va='bottom')
    plt.title('Sequence Length Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Sequence Length')
    plt.ylabel('Frequency')
    
    plt.tight_layout()
    plot_path = os.path.join(artifact_dir, 'data_distribution.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Total samples: {len(y)}")
    print(f"Number of classes: {len(label_map)}")
    print(f"Average samples per class: {len(y)/len(label_map):.1f}")

## 4. Training Pipeline with MLflow

In [ ]:
def train_model_with_visualization(data_dir, model_save_path, artifact_dir, epochs=50, batch_size=8):
    """Train the VideoMAE sign language recognition model with comprehensive visualization"""
    
    # Initialize MLflow
    mlflow.set_experiment("VideoMAE Sign Language Recognition")
    
    with mlflow.start_run() as run:
        # Enable auto-logging
        mlflow.tensorflow.autolog()
        
        # Log parameters following paper methodology
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("sequence_length", 64)
        mlflow.log_param("masking_ratio", 0.9)  # Paper uses 90-95%
        mlflow.log_param("artifact_dir", artifact_dir)
        
        # Process data following paper methodology
        print("Processing video data following paper methodology...")
        processor = SignLanguageDataProcessor(sequence_length=64)
        X, y, label_map = processor.create_dataset(data_dir)
        
        # Save label map
        label_map_path = os.path.join(artifact_dir, 'label_map.json')
        with open(label_map_path, 'w') as f:
            json.dump(label_map, f)
        mlflow.log_artifact(label_map_path)
        
        print(f"Dataset shape: {X.shape}")
        print(f"Number of classes: {len(label_map)}")
        print(f"Classes: {list(label_map.values())}")
        
        # Log dataset info
        mlflow.log_param("num_classes", len(label_map))
        mlflow.log_param("dataset_size", len(X))
        
        # Visualize data distribution
        visualize_data_distribution(y, label_map, artifact_dir)
        
        # Convert labels to categorical
        y_cat = keras.utils.to_categorical(y, num_classes=len(label_map))
        
        # Split data following paper's approach (4:1:1 ratio)
        # Paper: training+validation combined, test set used for both validation and testing
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y_cat, test_size=0.167, random_state=42, stratify=y
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=0.2, random_state=42, stratify=np.argmax(y_temp, axis=1)
        )
        
        print(f"Training set: {X_train.shape}")
        print(f"Validation set: {X_val.shape}")
        print(f"Test set: {X_test.shape}")
        
        # Build model following paper specifications
        model_builder = PaperCompliantVideoMAE(
            input_shape=(X.shape[1], X.shape[2]),
            num_classes=len(label_map),
            d_model=768,      # Paper uses 768 for base model
            num_heads=12,     # Paper uses 12 heads
            num_layers=12     # Paper uses 12 layers
        )
        model = model_builder.build_model()
        
        # Compile model following paper's optimizer
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # Paper uses this learning rate
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Log model summary
        model_summary_path = os.path.join(artifact_dir, 'model_summary.txt')
        with open(model_summary_path, 'w') as f:
            model.summary(print_fn=lambda x: f.write(x + '\n'))
        mlflow.log_artifact(model_summary_path)
        
        # Callbacks following paper's approach
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_accuracy',
                patience=10,
                restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7
            )
        ]
        
        # Train model following paper's methodology
        print("Training model following paper methodology...")
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
        
        # Plot training history
        plot_training_history(history, artifact_dir)
        
        # Evaluate model
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"Test Accuracy: {test_accuracy:.4f}")
        mlflow.log_metric("test_accuracy", test_accuracy)
        
        # Detailed evaluation
        detailed_evaluation(model, X_test, y_test, label_map, artifact_dir)
        
        # Save model
        model_save_path = os.path.join(artifact_dir, model_save_path)
        model.save(model_save_path)
        mlflow.log_artifact(model_save_path)
        
        print("Training completed successfully following paper methodology!")
        return model, history, run.info.run_id

## 5. Visualization Functions

In [ ]:
def plot_training_history(history, artifact_dir):
    """Plot training history with multiple metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Learning rate (if available)
    if 'lr' in history.history:
        axes[1, 0].plot(history.history['lr'], label='Learning Rate', color='green', linewidth=2)
        axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Learning Rate')
        axes[1, 0].set_yscale('log')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].axis('off')
    
    # Combined view
    axes[1, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[1, 1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[1, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1, 1].set_title('Combined Metrics', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Value')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(artifact_dir, 'training_history.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def detailed_evaluation(model, X_test, y_test, label_map, artifact_dir):
    """Perform detailed evaluation with visualizations"""
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    
    # Classification report
    class_names = [label_map[i] for i in range(len(label_map))]
    report = classification_report(y_true_classes, y_pred_classes, 
                                 target_names=class_names, output_dict=True)
    
    # Convert to DataFrame for better visualization
    report_df = pd.DataFrame(report).iloc[:-1, :].T
    
    # Plot classification report
    plt.figure(figsize=(12, 8))
    sns.heatmap(report_df.iloc[:-1, :3], annot=True, cmap='Blues', 
                cbar_kws={'label': 'Score'}, fmt='.3f')
    plt.title('Classification Report', fontsize=16, fontweight='bold')
    plt.xlabel('Metrics')
    plt.ylabel('Classes')
    plt.tight_layout()
    plot_path = os.path.join(artifact_dir, 'classification_report.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Confusion matrix
    cm = confusion_matrix(y_true_classes, y_pred_classes)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    cm_path = os.path.join(artifact_dir, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Per-class accuracy
    class_accuracy = cm.diagonal() / cm.sum(axis=1)
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(class_names)), class_accuracy, 
                   color=plt.cm.viridis(np.linspace(0, 1, len(class_names))))
    plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
    plt.xlabel('Classes')
    plt.ylabel('Accuracy')
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
    
    # Add value labels on bars
    for bar, acc in zip(bars, class_accuracy):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{acc:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    pca_path = os.path.join(artifact_dir, 'per_class_accuracy.png')
    plt.savefig(pca_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Top-5 accuracy (as mentioned in paper)
    top5_accuracy = top_k_accuracy(y_test, y_pred, k=5)
    print(f"Top-5 Accuracy: {top5_accuracy:.4f}")
    mlflow.log_metric("top5_accuracy", top5_accuracy)
    
    return report_df

In [ ]:
def top_k_accuracy(y_true, y_pred, k=5):
    """Calculate top-k accuracy"""
    top_k_preds = np.argsort(y_pred, axis=1)[:, -k:]
    top_k_correct = np.any(top_k_preds.T == np.argmax(y_true, axis=1), axis=0)
    return np.mean(top_k_correct)

## 6. Model Architecture Visualization

In [ ]:
def visualize_model_architecture(model, artifact_dir):
    """Visualize model architecture"""
    # Plot model architecture
    arch_path = os.path.join(artifact_dir, 'model_architecture.png')
    tf.keras.utils.plot_model(
        model,
        to_file=arch_path,
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        expand_nested=True,
        dpi=96
    )
    
    # Log to MLflow
    mlflow.log_artifact(arch_path)
    
    # Display the architecture
    from IPython.display import Image
    return Image(arch_path)

## 7. Main Training Execution

In [ ]:
def main():
    # Set your data directory here
    DATA_DIR = "path/to/your/sign_language_dataset"  # Update this path
    MODEL_SAVE_PATH = "videomae_slr_model.h5"
    
    # Check if data directory exists
    if not os.path.exists(DATA_DIR):
        print(f"Data directory {DATA_DIR} not found. Please update the path.")
        return
    
    # Train model
    model, history, run_id = train_model_with_visualization(
        data_dir=DATA_DIR,
        model_save_path=MODEL_SAVE_PATH,
        artifact_dir=artifact_dir,
        epochs=50,
        batch_size=8
    )
    
    print(f"\nTraining completed! Run ID: {run_id}")
    print(f"Model saved to: {os.path.join(artifact_dir, MODEL_SAVE_PATH)}")
    print(f"Artifacts saved to: {artifact_dir}")
    
    # Visualize model architecture
    try:
        model_viz = visualize_model_architecture(model, artifact_dir)
        display(model_viz)
    except Exception as e:
        print(f"Could not visualize model architecture: {e}")
    
    return model, history

## 8. Run Training

**Before running, please update the `DATA_DIR` variable in the `main()` function to point to your sign language dataset.**

Expected directory structure:
```
your_data/
├── SIGN_CLASS_1/
│   ├── video1.mp4
│   ├── video2.mov
│   └── ...
├── SIGN_CLASS_2/
│   ├── video1.mp4
│   ├── video2.mov
│   └── ...
└── ...
```

In [ ]:
# Execute training
# Uncomment the line below to run training
# model, history = main()

## 9. MLflow Experiment Tracking

To view the MLflow dashboard:

1. In your terminal, run: `mlflow ui`
2. Open your browser and go to: http://localhost:5000

You will see:
- Training parameters logged
- Metrics tracked over time
- Model artifacts saved
- Training history plots

## 10. Generated Artifacts

After training, the following files will be generated in a timestamped folder under `./artifacts/`:

1. `videomae_slr_model.h5` - Trained model
2. `label_map.json` - Class label mapping
3. `model_summary.txt` - Model architecture summary
4. `training_history.png` - Training/validation curves
5. `classification_report.png` - Detailed classification metrics
6. `confusion_matrix.png` - Confusion matrix visualization
7. `per_class_accuracy.png` - Per-class accuracy scores
8. `model_architecture.png` - Model architecture diagram
9. `data_distribution.png` - Dataset distribution visualization